In [126]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

def find_project_root(start: Path, marker: str = "src") -> Path:
    current = start.resolve()
    for path in [current, *current.parents]:
        if (path / marker).exists():
            return path
    raise RuntimeError(f"Could not find project root containing '{marker}'")

PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)

PROJECT_ROOT = C:\Users\rawls.varghese\quant-lab


In [127]:
from src.utils.metrics import sharpe_ratio, max_drawdown
from src.features.returns import make_forward_returns
from src.portfolio.construction import equal_weight_long_short
from src.backtest.engine import backtest_cross_sectional_strategy

print("Imports working")

Imports working


**Load The Feature Dataset**

In [128]:


from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error

df = pd.read_csv("../../../data/processed/project_04_features.csv")
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Date", "Ticker"]).copy()

df.head()

,Date,Ticker,ret_1,ret_5,ret_10,ret_20,vol_5,vol_20,ma_10_ratio,ma_20_ratio,ma_50_ratio,dollar_vol,vol_z_20,fwd_ret_5
0,1985-03-13,AAPL,-0.05579,-0.117225,-0.134128,-0.269486,0.058919,0.037152,-0.078848,-0.143323,-0.217203,2.430735e+07,-0.225229,0.022112
37427,1985-03-13,CVX,0.00000,-0.014464,-0.019368,-0.007230,0.011254,0.014927,-0.015044,-0.013709,0.038858,1.345384e+06,-1.613667,0.007282
59821,1985-03-13,HD,0.00000,-0.023534,0.000000,0.171662,0.010525,0.025004,-0.016688,0.007537,0.071796,2.615801e+05,-1.103125,0.000000
70144,1985-03-13,JNJ,0.00000,0.011303,0.000000,0.022387,0.006320,0.012735,0.002402,0.012644,0.040975,2.193474e+07,1.562101,0.016184
80468,1985-03-13,JPM,-0.00351,-0.020031,-0.029772,-0.050104,0.002799,0.017757,-0.020888,-0.029617,0.004920,3.122107e+06,-0.472377,0.006813


**Features and Target**

In [129]:
feature_cols = [
    "ret_1", "ret_5", "ret_10", "ret_20",
    "vol_5", "vol_20",
    "ma_10_ratio", "ma_20_ratio", "ma_50_ratio",
    "dollar_vol", "vol_z_20"
]

target_col = "fwd_ret_5"

X = df[feature_cols].copy()
y = df[target_col].copy()

**Time Split**

In [130]:
split_date = "2016-01-01"

train_mask = df["Date"] < split_date
test_mask = df["Date"] >= split_date

X_train = X.loc[train_mask]
X_test = X.loc[test_mask]

y_train = y.loc[train_mask]
y_test = y.loc[test_mask]

df_train = df.loc[train_mask].copy()
df_test = df.loc[test_mask].copy()

print('Train Shape', X_train.shape)
print("Test Shape", X_test.shape)
print("Train date range", df_train["Date"].min(), df_train["Date"].max())
print("Test Date Range", df_test["Date"].min(), df_test["Date"].max())

Train Shape (124361, 11)
Test Shape (51160, 11)
Train date range 1985-03-13 00:00:00 2015-12-31 00:00:00
Test Date Range 2016-01-04 00:00:00 2026-03-06 00:00:00


**Train baseline linear regression**

In [131]:
linreg = LinearRegression()
linreg.fit(X_train, y_train)

pred_train = linreg.predict(X_train)
pred_test = linreg.predict(X_test)

**Baseline Errors**

In [132]:
rmse_train = mean_squared_error(y_train, pred_train)
rmse_test = mean_squared_error(y_test, pred_test)

mae_train = mean_absolute_error(y_train, pred_train)
mae_test = mean_absolute_error(y_test, pred_test)

print("LinearRegression")
print("Train RMSE", rmse_train)
print("Test RMSE", rmse_test)
print("Train MAE", mae_train)
print("Test mae", mae_test ), 

LinearRegression
Train RMSE 0.0024896995740079472
Test RMSE 0.0018905840569979078
Train MAE 0.03317610677399117
Test mae 0.02977837671487088


(None,)

**Prediction Correlation**

In [133]:
df_test = df_test.copy()
df_test["pred"] = pred_test

test_corr = df_test["pred"].corr(df_test["fwd_ret_5"])
print("Test prediction correlation", test_corr)

Test prediction correlation -0.005014972867792463


The baseline linear regression model achieved low prediction error but near-zero out-of-sample correlation (≈ -0.005) between predicted and actual 5-day returns, indicating that the model currently has little ability to rank stocks by future performance.

**Cross-sectional Information Coefficient**

In [134]:
daily_ic = df_test.groupby("Date").apply(
    lambda x : x["pred"].corr(x["fwd_ret_5"])
    )

print(daily_ic.describe())
print("Mean daily IC:", daily_ic.mean())

count    2558.000000
mean       -0.023591
std         0.368667
min        -0.906190
25%        -0.303248
50%        -0.019674
75%         0.249872
max         0.935936
dtype: float64
Mean daily IC: -0.02359091729585502


**Decile / Quantile Test**

In [135]:
df_test["pred_q"] = df_test.groupby("Date")["pred"].transform(lambda x: pd.qcut(x, 5, labels=False, duplicates="drop"))


In [136]:
quantile_returns = df_test.groupby("pred_q")["fwd_ret_5"].mean()
print(quantile_returns)

pred_q
0    0.005988
1    0.004278
2    0.003809
3    0.003879
4    0.004704
Name: fwd_ret_5, dtype: float64


The model exhibits a negative mean IC (~ -0.024) and an inverted quintile structure, indicating that while the features capture a real signal (likely mean reversion), the model’s predictions are directionally reversed, suggesting that flipping the signal may reveal usable alpha.

**Flip the Signal**


In [137]:
df_test["pred_flipped"] = -df_test["pred"]

daily_ic_flip = df_test.groupby("Date").apply(lambda x:x["pred_flipped"].corr(x["fwd_ret_5"]))
print ("Mean Flipped IC:", daily_ic_flip.mean())

df_test["q_pred_flipped"] = df_test.groupby("Date")["pred_flipped"].transform(lambda x: pd.qcut(x, 5, labels=False, duplicates="drop"))
print(df_test.groupby("q_pred_flipped")["fwd_ret_5"].mean())

Mean Flipped IC: 0.02359091729585502
q_pred_flipped
0    0.004704
1    0.003879
2    0.003809
3    0.004278
4    0.005988
Name: fwd_ret_5, dtype: float64


After correcting the signal direction, the model achieves a positive mean daily IC of approximately 0.024, indicating a strong cross-sectional ranking ability. Quintile analysis confirms that higher predicted values correspond to higher future returns, validating the presence of a short-term mean reversion signal, although the relationship remains moderately noisy across intermediate buckets.

**Long short Portfolio**

In [138]:
daily_portfolio = df_test.groupby("Date").apply(
    lambda x: pd.Series({
        "long_ret": x.loc[x["q_pred_flipped"] == x["q_pred_flipped"].max(), "fwd_ret_5"].mean(),
        "short_ret": x.loc[x["q_pred_flipped"] == x["q_pred_flipped"].min(), "fwd_ret_5"].mean() 
     })
)

**Spread Return**

In [139]:
daily_portfolio["ls_ret_5d"] = daily_portfolio["long_ret"] - daily_portfolio["short_ret"]
daily_portfolio["ls_ret_daily"] = daily_portfolio["ls_ret_5d"] / 5

**Equity**

In [140]:
daily_portfolio["equity"] = (1 + daily_portfolio["ls_ret_daily"]).cumprod()

**Sharpe & Drawdown**

In [141]:
import os
print(os.getcwd())

c:\Users\rawls.varghese\quant-lab\research\project_04_return_forecast_alpha\notebooks


In [142]:
from src.utils.metrics import sharpe_ratio, max_drawdown

sharpe = sharpe_ratio(daily_portfolio["ls_ret_daily"])
dd = max_drawdown(daily_portfolio["equity"])

print(sharpe)
print(dd)

0.5293727705730397
-0.4822250188849363


The baseline model (v1) captured a weak but consistent mean-reversion signal that produced modest positive returns, though with significant volatility and drawdowns, indicating limited but potentially improvable trading value.

**SRC pipeline**

In [146]:
from src.signals.ranking import standardize_signal
from src.portfolio.construction import build_daily_weights_from_panel
from src.backtest.engine import backtest_cross_sectional_strategy

df_test = standardize_signal(df_test, "pred_flipped")

weights_df = build_daily_weights_from_panel(
    df_test,
    signal_col="signal_z",
    date_col="Date",
    asset_col="Ticker"
)

portfolio_df = backtest_cross_sectional_strategy(
    weights_df,
    df_test,
    return_col="fwd_ret_5"
)

In [ ]:
new_sharpe = sharpe_ratio(portfolio_df["portfolio_return"])

print("Old Sharpe:", sharpe)
print("New Sharpe:", new_sharpe) 

Old Sharpe: 0.5293727705730397
New Sharpe: 0.5293727705730397


In [149]:
output_path = PROJECT_ROOT / "data" / "processed" / "v1.csv"
df_test.to_csv(output_path, index=False)